# 🔁 LSTM Training v3 — Global Multi-country Electricity Forecasting

**FIX theo chuẩn trainTFT_v3:**
- ✅ Shift weather features 1 bước (Zero Leakage)
- ✅ Thêm feature: `log_precipitation`, `prec_zscore`, `temp_anomaly`, `solar_norm`, `yoy_change`, `roll_max_6`
- ✅ Loại `Other fossil` khỏi training (quá nhiều zeros)
- ✅ Checkpoint path nhất quán với TFT v3
- ✅ Metrics: MAE, RMSE, SMAPE, WAPE

---
| Mục | Thông tin |
|---|---|
| Dataset | `tft_premodel_dataset_EDA.csv` |
| Target series | Coal, Gas, Hydro, Solar, Wind, Bioenergy, Nuclear, Other Renewables |
| Encoder length | 24 tháng |
| Prediction length | 6 tháng |
| Checkpoint output | `checkpoint/lstm_v3_best.ckpt` |

In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — IMPORTS & DEVICE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os, json
from pathlib import Path
warnings.filterwarnings('ignore')

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import (
    EarlyStopping, ModelCheckpoint, LearningRateMonitor
)
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.models.rnn import RecurrentNetwork
from pytorch_forecasting.metrics import MAE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ PyTorch  : {torch.__version__}')
print(f'   Device   : {DEVICE}',
      f'({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else '')

BASE_DIR = Path(r"C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting")
DATA_PATH = BASE_DIR / "data" / "processed" / "tft_premodel_dataset_EDA.csv"
CKPT_DIR  = BASE_DIR / "checkpoint"
CKPT_DIR.mkdir(exist_ok=True)
print(f'   Data      : {DATA_PATH}')
print(f'   Checkpoint: {CKPT_DIR}')

✅ PyTorch  : 2.5.1+cu121
   Device   : cuda (NVIDIA GeForce RTX 3050 Ti Laptop GPU)
   Data      : C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\data\processed\tft_premodel_dataset_EDA.csv
   Checkpoint: C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — HYPERPARAMETERS (đồng bộ với TFT v3)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CFG = dict(
    # ── Dataset ───────────────────────────────────────────────────────────────
    max_encoder_length    = 24,
    max_prediction_length = 6,
    min_series_length     = 30,
    val_cutoff_months     = 12,

    # ── Model ─────────────────────────────────────────────────────────────────
    hidden_size           = 32,
    rnn_layers            = 2,
    dropout               = 0.1,

    # ── Training ──────────────────────────────────────────────────────────────
    learning_rate         = 1e-3,
    batch_size            = 64,
    max_epochs            = 80,
    gradient_clip_val     = 0.1,
    patience              = 12,
    num_workers           = 0,

    # ── Seed ──────────────────────────────────────────────────────────────────
    seed                  = 42,
)

pl.seed_everything(CFG['seed'], workers=True)
print('📋 Config LSTM v3:')
for k, v in CFG.items():
    print(f'  {k:<28}: {v}')

Seed set to 42


📋 Config LSTM v3:
  max_encoder_length          : 24
  max_prediction_length       : 6
  min_series_length           : 30
  val_cutoff_months           : 12
  hidden_size                 : 32
  rnn_layers                  : 2
  dropout                     : 0.1
  learning_rate               : 0.001
  batch_size                  : 64
  max_epochs                  : 80
  gradient_clip_val           : 0.1
  patience                    : 12
  num_workers                 : 0
  seed                        : 42


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — LOAD & PREPROCESSING (FIX: Zero Leakage + Feature Engineering)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
df_raw = pd.read_csv(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw['date'])
print(f'Raw shape: {df_raw.shape}')

# ── FIX #1: Chỉ giữ các series nguồn điện thực sự (loại Other fossil như TFT v3)
TARGET_SERIES = [
    'Coal', 'Gas', 'Hydro', 'Solar', 'Wind',
    'Bioenergy', 'Nuclear', 'Other Renewables',
    # 'Other Fossil' bị loại: 73% zeros, model không học được
]
df = df_raw[df_raw['series'].isin(TARGET_SERIES)].copy()
print(f'After series filter: {df.shape}')

# ── FIX #2: Shift weather features để tránh data leakage
# (Giống hệt trainTFT_v3 Cell 3)
weather_cols = ['temperature', 'solar', 'humidity', 'precipitation']
for col in weather_cols:
    if col in df.columns:
        df[col] = df.groupby(['entity', 'series'])[col].shift(1)
        df[col] = df.groupby(['entity', 'series'])[col].transform(
            lambda x: x.fillna(0)
        )
print('✅ Weather features shifted (Zero Leakage).')

# ── FIX #3: Thêm derived weather features (giống TFT v3)
# log_precipitation
if 'precipitation' in df.columns:
    df['log_precipitation'] = np.log1p(df['precipitation'].clip(lower=0))

# prec_zscore
mu  = df.groupby('entity')['precipitation'].transform('mean')
std = df.groupby('entity')['precipitation'].transform('std').replace(0, 1)
df['prec_zscore'] = (df['precipitation'] - mu) / std

# temp_anomaly: nhiệt độ lệch khỏi trung bình tháng
if 'temperature' in df.columns:
    df['month_tmp'] = df['date'].dt.month
    month_mean = df.groupby(['entity', 'series', 'month_tmp'])['temperature'].transform('mean')
    df['temp_anomaly'] = df['temperature'] - month_mean
    df.drop(columns=['month_tmp'], inplace=True)

# solar_norm: chuẩn hóa solar theo max của group
if 'solar' in df.columns:
    solar_max = df.groupby(['entity', 'series'])['solar'].transform('max').replace(0, 1)
    df['solar_norm'] = df['solar'] / solar_max

# prec_lag_1, prec_lag_2
if 'precipitation' in df.columns:
    df['prec_lag_1'] = df.groupby(['entity', 'series'])['precipitation'].shift(1).fillna(0)
    df['prec_lag_2'] = df.groupby(['entity', 'series'])['precipitation'].shift(2).fillna(0)

print('✅ Derived weather features created: log_precipitation, prec_zscore, temp_anomaly, solar_norm, prec_lag_1/2')

# ── FIX #4: Lag features cho target (giống TFT v3)
def create_lag_roll(group):
    g = group.copy().sort_values('date')
    g['gen_lag_1']  = g['generation_TWh'].shift(1)
    g['gen_lag_3']  = g['generation_TWh'].shift(3)
    g['gen_lag_12'] = g['generation_TWh'].shift(12)
    g['roll_mean_3']  = g['generation_TWh'].shift(1).rolling(3,  min_periods=1).mean()
    g['roll_mean_6']  = g['generation_TWh'].shift(1).rolling(6,  min_periods=1).mean()
    g['roll_std_3']   = g['generation_TWh'].shift(1).rolling(3,  min_periods=1).std().fillna(0)
    g['roll_max_6']   = g['generation_TWh'].shift(1).rolling(6,  min_periods=1).max()
    # yoy_change: % thay đổi so với cùng kỳ năm trước
    g['yoy_change'] = g['generation_TWh'].pct_change(12).fillna(0).replace([np.inf, -np.inf], 0)
    return g

df = df.groupby(['entity', 'series'], group_keys=False).apply(create_lag_roll)

# Fill NaN ở lag/roll
lag_cols = ['gen_lag_1','gen_lag_3','gen_lag_12',
            'roll_mean_3','roll_mean_6','roll_std_3','roll_max_6','yoy_change',
            'prec_lag_1','prec_lag_2']
for col in lag_cols:
    if col in df.columns:
        df[col] = df.groupby(['entity','series'])[col].transform(lambda x: x.fillna(0))
print('✅ Lag & rolling features created.')

# ── Time features ──────────────────────────────────────────────────────────────
df['month']      = df['date'].dt.month
df['quarter']    = df['date'].dt.quarter
df['year']       = df['date'].dt.year
df['month_sin']  = np.sin(2 * np.pi * df['date'].dt.month / 12)
df['month_cos']  = np.cos(2 * np.pi * df['date'].dt.month / 12)

# ── time_idx liên tục per group (giống TFT v3) ─────────────────────────────────
df = df.sort_values(['entity','series','date'])
df['time_idx'] = df.groupby(['entity','series'])['date'].rank(method='dense').astype(int) - 1

# ── Lọc series đủ độ dài ──────────────────────────────────────────────────────
counts = df.groupby(['entity','series']).size()
valid  = counts[counts >= CFG['min_series_length']].reset_index()[['entity','series']]
df     = df.merge(valid, on=['entity','series'])

print(f'✅ Preprocessing done. Total NaN: {df.isnull().sum().sum()}')
print(f'   Shape: {df.shape} | Groups: {df.groupby(["entity","series"]).ngroups}')

Raw shape: (34614, 30)
After series filter: (12197, 30)
✅ Weather features shifted (Zero Leakage).
✅ Derived weather features created: log_precipitation, prec_zscore, temp_anomaly, solar_norm, prec_lag_1/2
✅ Lag & rolling features created.
✅ Preprocessing done. Total NaN: 0
   Shape: (12178, 30) | Groups: 133


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — TRAIN / VAL SPLIT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
training_cutoff = int(df['time_idx'].max()) - CFG['val_cutoff_months']

print(f'📅 Train/Val Split:')
print(f'   training_cutoff (time_idx): {training_cutoff}')
print(f'   Train rows : {(df["time_idx"] <= training_cutoff).sum():,}')
print(f'   Val rows   : {(df["time_idx"] >  training_cutoff).sum():,}')

📅 Train/Val Split:
   training_cutoff (time_idx): 83
   Train rows : 10,943
   Val rows   : 1,235


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — TimeSeriesDataSet
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# QUAN TRỌNG VỚI LSTM:
# RecurrentNetwork yêu cầu TẤT CẢ features phải là known_reals
# (vì decoder cần nhìn thấy chúng ở cả encoder lẫn prediction window)
# → Tất cả features đưa vào known_reals, chỉ để target ở unknown_reals

TIME_VARYING_KNOWN_REALS = [
    'time_idx',
    'month', 'month_sin', 'month_cos', 'quarter', 'year',
    # Weather (đã shift → safe)
    'temperature', 'solar', 'humidity', 'precipitation',
    'log_precipitation', 'prec_zscore', 'temp_anomaly', 'solar_norm',
    'prec_lag_1', 'prec_lag_2',
    # Lag/roll target
    'gen_lag_1', 'gen_lag_3', 'gen_lag_12',
    'roll_mean_3', 'roll_mean_6', 'roll_std_3', 'roll_max_6',
    'yoy_change',
]

# Kiểm tra columns tồn tại
TIME_VARYING_KNOWN_REALS = [c for c in TIME_VARYING_KNOWN_REALS if c in df.columns]
TIME_VARYING_UNKNOWN_REALS = ['generation_TWh']

all_needed = TIME_VARYING_KNOWN_REALS + TIME_VARYING_UNKNOWN_REALS + ['entity', 'series']
missing = [c for c in all_needed if c not in df.columns]
if missing:
    print(f'⚠️  Thiếu columns: {missing}')
else:
    print('✅ Tất cả columns đều có mặt')

print('⏳ Đang tạo TimeSeriesDataSet...')
training = TimeSeriesDataSet(
    df[df['time_idx'] <= training_cutoff],
    time_idx                         = 'time_idx',
    target                           = 'generation_TWh',
    group_ids                        = ['entity', 'series'],
    min_encoder_length               = CFG['max_encoder_length'] // 2,
    max_encoder_length               = CFG['max_encoder_length'],
    min_prediction_length            = 1,
    max_prediction_length            = CFG['max_prediction_length'],
    static_categoricals              = ['entity'],
    static_reals                     = [],
    time_varying_known_categoricals  = ['series'],
    time_varying_known_reals         = TIME_VARYING_KNOWN_REALS,
    time_varying_unknown_categoricals= [],
    time_varying_unknown_reals       = TIME_VARYING_UNKNOWN_REALS,
    target_normalizer                = GroupNormalizer(
        groups=['entity', 'series'],
        transformation='softplus',
    ),
    add_relative_time_idx            = True,
    add_target_scales                = True,
    add_encoder_length               = True,
    allow_missing_timesteps          = True,
)

validation = TimeSeriesDataSet.from_dataset(
    training, df, predict=True, stop_randomization=True
)
print(f'✅ Training dataset  : {len(training):,} samples')
print(f'✅ Validation dataset: {len(validation):,} samples')

✅ Tất cả columns đều có mặt
⏳ Đang tạo TimeSeriesDataSet...
✅ Training dataset  : 11,608 samples
✅ Validation dataset: 133 samples


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — DATALOADERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
train_loader = training.to_dataloader(
    train=True, batch_size=CFG['batch_size'], num_workers=CFG['num_workers']
)
val_loader = validation.to_dataloader(
    train=False, batch_size=CFG['batch_size'] * 2, num_workers=CFG['num_workers']
)

# Sanity check
x, y = next(iter(train_loader))
print('✅ DataLoaders sẵn sàng')
print(f'   Encoder shape: {x["encoder_cont"].shape}  (batch, time, features)')
print(f'   Decoder shape: {x["decoder_cont"].shape}')
print(f'   Target shape : {y[0].shape}')

✅ DataLoaders sẵn sàng
   Encoder shape: torch.Size([64, 24, 29])  (batch, time, features)
   Decoder shape: torch.Size([64, 6, 29])
   Target shape : torch.Size([64, 6])


In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — BUILD LSTM MODEL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
model = RecurrentNetwork.from_dataset(
    training,
    cell_type     = 'LSTM',
    hidden_size   = CFG['hidden_size'],
    rnn_layers    = CFG['rnn_layers'],
    dropout       = CFG['dropout'],
    learning_rate = CFG['learning_rate'],
    loss          = MAE(),
    reduce_on_plateau_patience = 4,
    log_interval  = 10,
    log_val_interval = 1,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ LSTM Model built')
print(f'   Parameters : {n_params:,}')
print(f'   Hidden size: {CFG["hidden_size"]}')
print(f'   RNN layers : {CFG["rnn_layers"]}')
print(f'   Cell type  : LSTM')

✅ LSTM Model built
   Parameters : 18,557
   Hidden size: 32
   RNN layers : 2
   Cell type  : LSTM


In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — TRAINER & CALLBACKS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
early_stop = EarlyStopping(
    monitor='val_loss', min_delta=1e-4,
    patience=CFG['patience'], mode='min'
)
lr_logger = LearningRateMonitor()
ckpt_cb = ModelCheckpoint(
    dirpath   = str(CKPT_DIR),
    monitor   = 'val_loss',
    mode      = 'min',
    save_top_k= 1,
    filename  = 'lstm_v3_best',   # → lstm_v3_best.ckpt
)

trainer = pl.Trainer(
    max_epochs        = CFG['max_epochs'],
    accelerator       = 'gpu' if torch.cuda.is_available() else 'cpu',
    devices           = 1,
    gradient_clip_val = CFG['gradient_clip_val'],
    callbacks         = [early_stop, lr_logger, ckpt_cb],
    log_every_n_steps = 10,
)

print('✅ Trainer ready. Bắt đầu training...')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


✅ Trainer ready. Bắt đầu training...


In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 9 — TRAIN
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
best_ckpt = ckpt_cb.best_model_path
print(f'✅ Training xong. Best checkpoint: {best_ckpt}')

You are using a CUDA device ('NVIDIA GeForce RTX 3050 Ti Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss             │ MAE            │      0 │ train │     0 │
│ 1 │ logging_metrics  │ ModuleList     │      0 │ train │     0 │
│ 2 │ embeddings       │ MultiEmbedding │    220 │ train │     0 │
│ 3 │ rnn              │ LSTM           │ 18.3 K │ train │     0 │
│ 4 │ output_projector │ Linear         │     33 │ train │     0 │
└───┴──────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 18.6 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 18.6 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=80` reached.


✅ Training xong. Best checkpoint: C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint\lstm_v3_best-v1.ckpt


In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 10 — EVALUATION (FIX: thêm SMAPE, WAPE như TFT v3)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
best_model = RecurrentNetwork.load_from_checkpoint(best_ckpt)
best_model.eval()

preds = best_model.predict(val_loader, mode='prediction')
actuals = torch.cat([y[0] for _, y in val_loader])

y_pred = preds.detach().cpu().numpy().flatten()
y_true = actuals.detach().cpu().numpy().flatten()

n = min(len(y_true), len(y_pred))
y_true, y_pred = y_true[:n], y_pred[:n]
mask = np.isfinite(y_true) & np.isfinite(y_pred)
y_true, y_pred = y_true[mask], y_pred[mask]

mae   = mean_absolute_error(y_true, y_pred)
rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
r2    = r2_score(y_true, y_pred)
smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
wape  = np.abs(y_true - y_pred).sum() / (np.abs(y_true).sum() + 1e-8) * 100
mape  = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-4))) * 100

print('╔══════════════════════════════════════════════════════════════╗')
print('║                   KẾT QUẢ LSTM GLOBAL v3                    ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  MAE   : {mae:.4f} TWh                                      ║')
print(f'║  RMSE  : {rmse:.4f} TWh                                     ║')
print(f'║  MAPE  : {mape:.2f}%                                        ║')
print(f'║  SMAPE : {smape:.2f}%                                       ║')
print(f'║  WAPE  : {wape:.2f}%                                        ║')
print(f'║  R²    : {r2:.4f}                                           ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Best checkpoint: lstm_v3_best.ckpt                         ║')
print('╚══════════════════════════════════════════════════════════════╝')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


╔══════════════════════════════════════════════════════════════╗
║                   KẾT QUẢ LSTM GLOBAL v3                    ║
╠══════════════════════════════════════════════════════════════╣
║  MAE   : 0.2098 TWh                                      ║
║  RMSE  : 0.3746 TWh                                     ║
║  MAPE  : 2340.47%                                        ║
║  SMAPE : 17.19%                                       ║
║  WAPE  : 3.43%                                        ║
║  R²    : 0.9977                                           ║
╠══════════════════════════════════════════════════════════════╣
║  Best checkpoint: lstm_v3_best.ckpt                         ║
╚══════════════════════════════════════════════════════════════╝


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 11 — LƯU CONFIG (để LSTM_transfer_learning_v3 đọc lại)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
lstm_config = {
    'hidden_size'              : CFG['hidden_size'],
    'rnn_layers'               : CFG['rnn_layers'],
    'dropout'                  : CFG['dropout'],
    'max_encoder_length'       : CFG['max_encoder_length'],
    'max_prediction_length'    : CFG['max_prediction_length'],
    'target'                   : 'generation_TWh',
    'group_ids'                : ['entity', 'series'],
    'target_series'            : TARGET_SERIES,
    'time_varying_known_reals' : TIME_VARYING_KNOWN_REALS,
    'time_varying_unknown_reals': TIME_VARYING_UNKNOWN_REALS,
}

cfg_path = CKPT_DIR / 'lstm_v3_config.json'
with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(lstm_config, f, indent=2, ensure_ascii=False)
print(f'✅ Config saved → {cfg_path}')
print()
print('NEXT STEPS — Transfer Learning sang VN:')
print('  1. Mở LSTM_transfer_learning_v3.ipynb')
print('  2. Load checkpoint/lstm_v3_best.ckpt')
print('  3. Phase 1: Freeze rnn → fine-tune head (LR=3e-4, 15 epochs)')
print('  4. Phase 2: Unfreeze all → fine-tune với LR=3e-5 (30 epochs)')

✅ Config saved → C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint\lstm_v3_config.json

NEXT STEPS — Transfer Learning sang VN:
  1. Mở LSTM_transfer_learning_v3.ipynb
  2. Load checkpoint/lstm_v3_best.ckpt
  3. Phase 1: Freeze rnn → fine-tune head (LR=3e-4, 15 epochs)
  4. Phase 2: Unfreeze all → fine-tune với LR=3e-5 (30 epochs)


: 